# TP02: OOP & Design Patterns: Building a CSV Cleaning Project

**Goal:** build a small data-cleaning application for a Data Science project while
practicing Object-Oriented Programming and the classic design patterns.

| Exercise | Topic | Pattern |
|---|---|---|
| 1 | `CSVReader` class — encapsulation, constructor, methods | OOP basics |
| 2 | Interchangeable ways to handle missing values | **Strategy** |
| 3 | Logging and timing around existing methods | **Decorator** |
| 4 | Creating transformation objects from a string name | **Factory** |
| 5 | One consistent cleaning workflow | **Template Method** |

Each exercise builds on the previous one, so the notebook must be run from top to bottom.

## 0. Imports and configuration

Everything the notebook needs: `pandas` for the data, `abc` for the abstract base classes,
`functools.wraps` for the decorators, and `logging` / `time` for the cross-cutting concerns.

In [1]:
from __future__ import annotations

import logging
import time
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(message)s")
LOGGER = logging.getLogger("Data Cleaning Pipeline")

DATA_DIR = Path("/mnt/d/advanced-programming/src/data")
INPUT_PATH = DATA_DIR / "sample_data.csv"
OUTPUT_PATH = DATA_DIR / "cleaned_data.csv"

## Exercise 1: Build the `CSVReader` class (OOP foundation)

**Objective:** reinforce OOP fundamentals classes, constructors, methods, encapsulation.

**Concept:** all logic related to reading a CSV file lives inside one class, so the rest of
the program only has to call `read()` and `preview()` without knowing how the file is opened.

**Instructions**
- Create a class `CSVReader`.
- `__init__(self, file_path)` stores the path and prepares an empty `df` attribute.
- `read()` loads the file with pandas and returns a `DataFrame`.
- `preview(n)` prints the top `n` rows.

Encapsulation check: `self.df` starts as `None`, so `preview()` can warn the user instead of
crashing when it is called before `read()`.

In [2]:
class CSVReader:

    def __init__(self, file_path: Path) -> None:
        self.file_path = file_path
        self.df: pd.DataFrame | None = None

    def read(self) -> pd.DataFrame:
        
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int = 5) -> None:
        
        if self.df is None:
            LOGGER.warning("No data loaded - call read() first.")
            return
        
        LOGGER.info("Preview of %s (first %d rows):\n%s",
                    self.file_path.name, n, self.df.head(n)) 

In [3]:
reader = CSVReader(INPUT_PATH)
df = reader.read()

df.head()

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


In [4]:
LOGGER.info("Shape: %s", df.shape)
LOGGER.info("Missing values per column:\n%s", df.isna().sum())

Shape: (7, 7)
Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


## Exercise 2: Handle missing values with the Strategy Pattern

**Objective:** learn the Strategy Pattern — one interface, many interchangeable behaviours.
Practise inheritance and polymorphism.

**Concept:** the *what* (handling missing values) stays the same, while the *how* changes
(drop, fill mean, fill mode...). The client code never changes, only the object we inject.

**Instructions**
- Abstract class `MissingValueStrategy` with the method `handle(df)`.
- Subclasses `DropMissing`, `FillMean`, `FillMode`.
- `DataCleaner` receives a strategy and applies it.

`DataCleaner.clean()` is written once and works for every strategy: this is polymorphism.
A copy of the DataFrame is passed so the original data is never destroyed.

In [5]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        ...


class DropMissing(MissingValueStrategy):

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        dropped = int(df.isna().any(axis=1).sum())
        LOGGER.info("Dropping %d incomplete row(s).", dropped)
        return df.dropna()


class FillMean(MissingValueStrategy):

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        numeric = df.select_dtypes(include="number").columns
        df[numeric] = df[numeric].fillna(df[numeric].mean())
        return df


class FillMode(MissingValueStrategy):

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        for column in df.columns:
            most_common = df[column].mode()
            if not most_common.empty:
                df[column] = df[column].fillna(most_common.iloc[0])
        return df

In [6]:
class DataCleaner:

    def __init__(self, strategy: MissingValueStrategy) -> None:
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        LOGGER.info("Strategy in use: %s", type(self.strategy).__name__)
        return self.strategy.handle(df.copy())

In [7]:
strategies = {
    "Drop": DropMissing(),
    "Fill Mean": FillMean(),
    "Fill Mode": FillMode(),
}

for name, strategy in strategies.items():
    LOGGER.info("[INFO]: Using strategy %s \n", name)
    cleaned = DataCleaner(strategy).clean(df)
    LOGGER.info("[INFO]: Rows: %d to %d | missing left: %d",
                len(df), len(cleaned), int(cleaned.isna().sum().sum()))

[INFO]: Using strategy Drop 

Strategy in use: DropMissing
Dropping 5 incomplete row(s).
[INFO]: Rows: 7 to 2 | missing left: 0
[INFO]: Using strategy Fill Mean 

Strategy in use: FillMean
[INFO]: Rows: 7 to 7 | missing left: 0
[INFO]: Using strategy Fill Mode 

Strategy in use: FillMode
[INFO]: Rows: 7 to 7 | missing left: 0


## Exercise 3: Decorators for logging and timing

**Objective:** learn the Decorator Pattern to add reusable functionality (logging, timing)
and understand how decorators support the **Open/Closed Principle**.

**Concept:** a decorator wraps a function and extends its behaviour without modifying the
original code. The wrapped function is never touched, so it stays closed for modification —
we can add behaviour from the outside (open for extension).

**Instructions**
- `@log_action` — logs when a method starts and finishes.
- `@log_time` — calculates and prints the execution time.
- Apply them to the methods of `CSVReader` and `DataCleaner`.

Because the methods already exist (Exercises 1 and 2), we wrap them afterwards and assign
them back to the class. The method bodies stay exactly the same, which is the whole point of
the pattern.

In [8]:
def log_action(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        LOGGER.info("\n%s started", func.__qualname__)
        result = func(*args, **kwargs)
        LOGGER.info("\n%s finished", func.__qualname__)
        return result

    return wrapper


def log_time(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            LOGGER.info("[INFO]: %s took %.4fs", func.__qualname__,
                         time.perf_counter() - start)

    return wrapper

In [9]:
CSVReader.read = log_action(log_time(CSVReader.read))
CSVReader.preview = log_action(CSVReader.preview)
DataCleaner.clean = log_action(log_time(DataCleaner.clean))

LOGGER.info("CSVReader %s", CSVReader.read.__qualname__)
LOGGER.info("wrapped in: %s", CSVReader.read.__wrapped__.__qualname__)

CSVReader(INPUT_PATH).read()

CSVReader CSVReader.read
wrapped in: CSVReader.read

CSVReader.read started
[INFO]: CSVReader.read took 0.0039s

CSVReader.read finished


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## Exercise 4: Factory Pattern for data transformations

**Objective:** practise the Factory Pattern for scalable creation of transformation objects,
and apply abstraction and composition.

**Concept:** instead of writing `NormalizeColumns()` everywhere, we ask a factory for the
object by name. Adding a new transformation means adding one class and one registry entry —
the client code never changes.

**Instructions**
- Abstract class `DataTransform` with the method `apply(df)`.
- Subclasses `NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`.
- `TransformFactory` returns a transformation object from a string.

`NormalizeColumns` applies min-max scaling `(x - min) / (max - min)`, which puts every
numeric column between 0 and 1 — useful before a machine-learning model.

In [20]:
class DataTransform(ABC):

    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class NormalizeColumns(DataTransform):

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        for column in df.select_dtypes(include="number").columns:
            low, high = df[column].min(), df[column].max()
            if high != low:   # avoid dividing by zero
                df[column] = (df[column] - low) / (high - low)
        return df


class RemoveDuplicates(DataTransform):

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        LOGGER.info("Rows: %d -> %d", len(df), len(df.drop_duplicates()))
        return df.drop_duplicates()


class StandardizeText(DataTransform):
    
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        for column in df.select_dtypes(exclude="number").columns:
            df[column] = df[column].str.strip().str.lower()
        return df

In [21]:
class TransformFactory:

    _registry: dict[str, type[DataTransform]] = {
        "NormalizeColumns": NormalizeColumns,
        "RemoveDuplicates": RemoveDuplicates,
        "StandardizeText": StandardizeText,
    }

    @classmethod
    def create(cls, name: str) -> DataTransform:
        try:
            return cls._registry[name]()
        except KeyError:
            known = ", ".join(sorted(cls._registry))
            raise ValueError(
                f"Unknown transform {name!r}. Choose from: {known}"
            ) from None

In [23]:
for name in ("RemoveDuplicates", "StandardizeText", "NormalizeColumns"):
    transform = TransformFactory.create(name)
    LOGGER.info("[INFO]: Using transform %s -> %s", name, type(transform).__name__)

try:
    TransformFactory.create("unknown_transform")
except ValueError as error:
    LOGGER.warning("Factory error: %s", error)

[INFO]: Using transform RemoveDuplicates -> RemoveDuplicates
[INFO]: Using transform StandardizeText -> StandardizeText
[INFO]: Using transform NormalizeColumns -> NormalizeColumns
Factory error: Unknown transform 'unknown_transform'. Choose from: NormalizeColumns, RemoveDuplicates, StandardizeText


In [25]:
transformed = DataCleaner(FillMean()).clean(df)
for name in ("StandardizeText", "RemoveDuplicates", "NormalizeColumns"):
    transformed = TransformFactory.create(name).apply(transformed)

transformed.head()


DataCleaner.clean started
Strategy in use: FillMean
[INFO]: DataCleaner.clean took 0.0114s

DataCleaner.clean finished
Rows: 7 -> 7


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.25000,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,0.333333,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,0.500000,david,0.40625,0.466667,0.833333,houston,0.000000
4,0.666667,eva,0.00000,0.000000,0.000000,new york,0.764706


In [26]:
class DataPipeline(ABC):
    
    def __init__(self, source: Path, destination: Path) -> None:
        self.source = source
        self.destination = destination
        self.df: pd.DataFrame | None = None

    @abstractmethod
    def load(self) -> None: ...

    @abstractmethod
    def clean(self) -> None: ...

    @abstractmethod
    def transform(self) -> None: ...

    @abstractmethod
    def save(self) -> None: ...

    @log_action
    @log_time
    def run(self) -> None:
        LOGGER.info("PIPELINE START: %s to %s", self.source, self.destination)
        self.load()
        self.clean()
        self.transform()
        self.save()
        LOGGER.info("PIPELINE END: result saved in %s", self.destination)

In [27]:
class CSVDataPipeline(DataPipeline):

    def __init__(self, source: Path, destination: Path,
                 cleaner: DataCleaner, transforms: list[DataTransform]) -> None:
        super().__init__(source, destination)
        self.reader = CSVReader(source)
        self.cleaner = cleaner
        self.transforms = list(transforms)

    def load(self) -> None:
        self.df = self.reader.read()
        self.reader.preview(3)

    def clean(self) -> None:
        self.df = self.cleaner.clean(self.df)

    def transform(self) -> None:
        for transform in self.transforms:
            LOGGER.info("Transform: %s", type(transform).__name__)
            self.df = transform.apply(self.df)

    def save(self) -> None:
        self.df.to_csv(self.destination, index=False)

In [16]:
def build_pipeline(strategy: MissingValueStrategy, transform_names: list[str]) -> CSVDataPipeline:
    return CSVDataPipeline(
        source=INPUT_PATH,
        destination=OUTPUT_PATH,
        cleaner=DataCleaner(strategy),
        transforms=[TransformFactory.create(name) for name in transform_names],
    )

In [17]:
pipeline = build_pipeline(
    strategy=FillMean(),
    transform_names=["remove_duplicates", "standardize_text", "normalize"],
)

LOGGER.info("[INFO]: Collaborators of the pipeline:")
LOGGER.info("  reader     : %s", type(pipeline.reader).__name__)
LOGGER.info("  cleaner    : %s to %s", type(pipeline.cleaner).__name__,
            type(pipeline.cleaner.strategy).__name__)
LOGGER.info("  transforms : %s",
            [type(t).__name__ for t in pipeline.transforms])

pipeline.run()

[INFO]: Collaborators of the pipeline:
  reader     : CSVReader
  cleaner    : DataCleaner to FillMean
  transforms : ['RemoveDuplicates', 'StandardizeText', 'NormalizeColumns']

DataPipeline.run started
PIPELINE START: /mnt/d/advanced-programming/src/data/sample_data.csv to /mnt/d/advanced-programming/src/data/cleaned_data.csv

CSVReader.read started
[INFO]: CSVReader.read took 0.0044s

CSVReader.read finished

CSVReader.preview started
Preview of sample_data.csv (first 3 rows):
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN

CSVReader.preview finished

DataCleaner.clean started
Strategy in use: FillMean
[INFO]: DataCleaner.clean took 0.0073s

DataCleaner.clean finished
Transform: RemoveDuplicates
Rows: 7 -> 7
Transform: StandardizeText
Transform: NormalizeColumns
PIPELINE END: resul

In [18]:
build_pipeline(strategy=DropMissing(),
               transform_names=["standardize_text"]).run()


DataPipeline.run started
PIPELINE START: /mnt/d/advanced-programming/src/data/sample_data.csv to /mnt/d/advanced-programming/src/data/cleaned_data.csv

CSVReader.read started
[INFO]: CSVReader.read took 0.0049s

CSVReader.read finished

CSVReader.preview started
Preview of sample_data.csv (first 3 rows):
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN

CSVReader.preview finished

DataCleaner.clean started
Strategy in use: DropMissing
Dropping 5 incomplete row(s).
[INFO]: DataCleaner.clean took 0.0050s

DataCleaner.clean finished
Transform: StandardizeText
PIPELINE END: result saved in /mnt/d/advanced-programming/src/data/cleaned_data.csv
[INFO]: DataPipeline.run took 0.0447s

DataPipeline.run finished


In [19]:
LOGGER.info("RAW:\n%s", df)
LOGGER.info("CLEANED:\n%s", pd.read_csv(OUTPUT_PATH))

RAW:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0
CLEANED:
   id   name   age  height_cm  weight_kg      city  score
0   1  alice  29.0      165.0       68.0  new york   85.0
1   5    eva  27.0      160.0       55.0  new york   88.0
